In [ ]:
!git -C /content/a-triple-of-lms pull origin main

# Chạy Phi-3 sinh output trên Colab

Notebook này chỉ sinh câu trả lời Big Data cho 2 model Phi-3 local từ Hugging Face cache trên Google Drive. Notebook chỉ tạo output thô; phần đánh giá để chạy sau bằng script riêng.

Luồng chạy ngắn gọn:

1. Mount Google Drive.
2. Clone repo `Hutaph/a-triple-of-lms` vào Colab.
3. Trỏ cache Hugging Face tới thư mục Drive đang chứa model.
4. Chạy `src/phi3.py` trên `data/bigdata_10_questions.json` để lấy câu trả lời.
5. Lưu JSON output thô vào thư mục `output/` local của repo rồi xem nhanh output mẫu.

Mặc định notebook dùng `LOAD_IN_4BIT = True` để hợp với Colab T4. Nếu GPU nhiều VRAM hơn, có thể tắt 4-bit.

## 1. Mount Google Drive

Drive được dùng để đọc model đã cache. Output thô được lưu local trong `output/` của repo Colab. Nếu thư mục cache khác `/content/drive/MyDrive/hf_cache`, chỉnh biến `HF_CACHE_ROOT` ở cell cấu hình.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Cấu hình

Hai model mặc định:

- `microsoft/Phi-3-mini-4k-instruct`
- `microsoft/Phi-3-medium-4k-instruct`

Notebook chạy từ cache bằng `LOCAL_FILES_ONLY = True`, nên cần cache đã có sẵn trong Drive. Cấu trúc thường gặp là `hf_cache/hub/models--microsoft--...`. Mini và Medium dùng native Transformers.


In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/Hutaph/a-triple-of-lms.git"
PROJECT_DIR = Path("/content/a-triple-of-lms")

DRIVE_ROOT = Path("/content/drive/MyDrive")
HF_CACHE_ROOT = DRIVE_ROOT / "hf_cache"
HF_HUB_CACHE = HF_CACHE_ROOT / "hub"
CACHE_DIR_FOR_SCRIPT = HF_HUB_CACHE if HF_HUB_CACHE.exists() else HF_CACHE_ROOT

DATA_FILE = PROJECT_DIR / "data" / "bigdata_10_questions.json"
OUTPUT_DIR = PROJECT_DIR / "output"

MODELS_TO_RUN = ["mini", "medium"]  # chay 2 model local
OUTPUT_LIMIT = 0         # 0 = sinh output du 10 cau; dat 2 hoac 3 de smoke test nhanh
LOAD_IN_4BIT = True      # khuyen nghi cho Colab T4
LOCAL_FILES_ONLY = True  # dung model trong Drive cache, khong tai moi tu Hugging Face

os.environ["HF_HOME"] = str(HF_CACHE_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_ROOT / "transformers")
os.environ["PHI3_HF_CACHE_DIR"] = str(HF_HUB_CACHE)

print("Project dir:", PROJECT_DIR)
print("HF cache root:", HF_CACHE_ROOT)
print("HF hub cache:", HF_HUB_CACHE)
print("Cache dir for script:", CACHE_DIR_FOR_SCRIPT)
print("Output dir:", OUTPUT_DIR)
print("Models to run:", ", ".join(MODELS_TO_RUN))

## 3. Clone hoặc cập nhật repo

Cell này lấy `data` và `src` mới nhất từ GitHub. Nếu repo đã tồn tại trong runtime Colab, notebook chỉ `git pull`.

In [ ]:
import subprocess

if PROJECT_DIR.exists():
    print("Repo đã tồn tại, đang cập nhật...")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)
else:
    print("Đang clone repo...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())

if not (PROJECT_DIR / "src" / "phi3.py").exists():
    raise FileNotFoundError("src/phi3.py not found. Push the new file to GitHub or use a repo version that contains it.")

## 4. Cài thư viện cần thiết

Colab thường đã có PyTorch. Cell này cài thêm Transformers, Accelerate và BitsAndBytes cho Mini/Medium.

In [ ]:
!pip -q install -U "transformers>=4.43" accelerate bitsandbytes huggingface_hub tqdm python-dotenv tiktoken triton

## 5. Kiểm tra GPU và cache model

Nếu dòng cache báo `MISSING`, hãy kiểm tra lại `HF_CACHE_ROOT` hoặc tải model vào Drive trước. Notebook đang cố tình dùng cache local để tránh tải lại nhiều GB mỗi lần reset runtime.

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

MODEL_IDS = {
    "mini": "microsoft/Phi-3-mini-4k-instruct",
    "medium": "microsoft/Phi-3-medium-4k-instruct",
}

cache_base = HF_HUB_CACHE if HF_HUB_CACHE.exists() else HF_CACHE_ROOT
for key, model_id in MODEL_IDS.items():
    cache_name = "models--" + model_id.replace("/", "--")
    cache_path = cache_base / cache_name
    status = "OK" if cache_path.exists() else "MISSING"
    print(f"{key:10s} {status:7s} {cache_path}")

print("Models configured:", ", ".join(MODELS_TO_RUN))

## 6. Sinh output Phi-3

Cell này chỉ gọi model để sinh câu trả lời và lưu JSON output thô. Không truyền `--score` và không tạo summary đánh giá. Với T4, model medium có thể chậm; hãy đặt `OUTPUT_LIMIT = 2` để thử nhanh trước khi chạy đủ.

In [ ]:
import json
import shutil
import subprocess
from src.phi3 import safe_filename, save_results

TEMP_OUTPUT_DIR = OUTPUT_DIR / "_tmp_phi3_outputs"
if TEMP_OUTPUT_DIR.exists():
    shutil.rmtree(TEMP_OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_outputs = []

for model_key in MODELS_TO_RUN:
    model_output_dir = TEMP_OUTPUT_DIR / model_key
    cmd = [
        "python", "src/phi3.py",
        "--model", model_key,
        "--data", str(DATA_FILE),
        "--output-dir", str(model_output_dir),
        "--cache-dir", str(CACHE_DIR_FOR_SCRIPT),
        "--device", "auto",
        "--torch-dtype", "auto",
    ]

    if OUTPUT_LIMIT:
        cmd += ["--limit", str(OUTPUT_LIMIT)]
    if LOAD_IN_4BIT:
        cmd += ["--load-in-4bit"]
    if LOCAL_FILES_ONLY:
        cmd += ["--local-files-only"]

    print(f"\nGenerating outputs for {model_key}:")
    print(" ".join(cmd))
    completed = subprocess.run(cmd, text=True, capture_output=True)

    print("\nSTDOUT:")
    print(completed.stdout)

    if completed.stderr:
        print("\nSTDERR:")
        print(completed.stderr)

    completed.check_returncode()
    raw_output_path = next(model_output_dir.glob("phi3_*_outputs.json"))
    model_rows = json.loads(raw_output_path.read_text(encoding="utf-8"))
    all_outputs.extend(model_rows)

save_results(all_outputs, OUTPUT_DIR / "phi3_outputs.json")
for model_name in sorted({row["model_name"] for row in all_outputs}):
    model_rows = [row for row in all_outputs if row["model_name"] == model_name]
    save_results(model_rows, OUTPUT_DIR / f"{safe_filename(model_name)}_outputs.json")
shutil.rmtree(TEMP_OUTPUT_DIR, ignore_errors=True)

## 7. Xem lỗi theo model

Nếu `error` xuất hiện ở toàn bộ câu hỏi của một model, model thường đã lỗi ngay ở bước load. Cell này gom các lỗi duy nhất để biết nguyên nhân thật.

In [ ]:
import pandas as pd

outputs_path = OUTPUT_DIR / "phi3_outputs.json"
rows = json.loads(outputs_path.read_text(encoding="utf-8"))
outputs_df = pd.DataFrame(rows)

error_df = (
    outputs_df[["model_name", "error"]]
    .dropna()
    .drop_duplicates()
    .sort_values("model_name")
)
error_df

## 8. Xem nhanh output mẫu

Cell dưới đây đọc file tổng `phi3_outputs.json` và hiển thị vài cột chính để kiểm tra câu trả lời, lỗi load/generate, latency và số token output.

In [ ]:
cols = [
    "sample_id", "model_name", "difficulty", "category",
    "latency_s", "output_tokens", "error", "model_output",
]
available_cols = [column for column in cols if column in outputs_df.columns]
outputs_df[available_cols].head(10)

## 9. Đường dẫn kết quả

Các file chính sau khi chạy:

- `phi3_outputs.json`: toàn bộ output của Mini và Medium.
- `Phi-3_Mini_4K_outputs.json`, `Phi-3_Medium_4K_outputs.json`: output theo từng model.

In [ ]:
print("Output directory:", OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.glob("*.json")):
    print(path)